# SN32 DeBERTa Fine-Tuning for AI Detection
## Using ahmadreza13/human-vs-Ai-generated-dataset

This notebook fine-tunes DeBERTa-v3-large for the Bittensor SN32 AI detection subnet.

## 1. Install Dependencies

In [3]:
!pip install torch transformers datasets peft accelerate scikit-learn wandb -q
!pip install bitsandbytes -q  # For 8-bit quantization if needed

In [65]:
import sys
import subprocess

def install_package(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# Uninstall problematic packages
packages = ["transformers", "accelerate", "peft", "tokenizers"]
for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", pkg, "-y"], capture_output=True)

# Clear cache
subprocess.run([sys.executable, "-m", "pip", "cache", "purge"], capture_output=True)

# Install working versions
install_package("transformers==4.36.0")
install_package("accelerate==0.25.0")
install_package("peft==0.7.1")
install_package("tokenizers==0.15.0")

print("✓ Packages reinstalled")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 2.6 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 2.5 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [transformers] [transformers]


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
detection 3.14.0 requires accelerate==1.0.1, which is not installed.
detection 3.14.0 requires scalecodec, which is not installed.
detection 3.14.0 requires bittensor==10.0.1, but you have bittensor 10.3.2 which is incompatible.
detection 3.14.0 requires bittensor-cli==9.17.0, but you have bittensor-cli 9.21.2 which is incompatible.
detection 3.14.0 requires Requests==2.31.0, but you have requests 2.34.2 which is incompatible.
detection 3.14.0 requires setuptools==70.0.0, but you have setuptools 81.0.0 which is incompatible.
detection 3.14.0 requires torch==2.8.0, but you have torch 2.12.0 which is incompatible.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
detection 3.14.0 requires scalecodec, which is not installed.
detection 3.14.0 requires accelerate==1.0.1, but you have accelerate 0.25.0 which is incompatible.
detection 3.14.0 requires bittensor==10.0.1, but you have bittensor 10.3.2 which is incompatible.
detection 3.14.0 requires bittensor-cli==9.17.0, but you have bittensor-cli 9.21.2 which is incompatible.
detection 3.14.0 requires Requests==2.31.0, but you have requests 2.34.2 which is incompatible.
detection 3.14.0 requires setuptools==70.0.0, but you have setuptools 81.0.0 which is incompatible.
detection 3.14.0 requires torch==2.8.0, but you have torch 2.12.0 which is incompatible.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 2.0 MB/s  0:00:02 eta 0:00:01
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.15.2
    Uninstalling tokenizers-0.15.2:
      Successfully uninstalled tokenizers-0.15.2
✓ Packages reinstalled


## 2. Import Libraries

In [66]:
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
import os
import json
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version: 2.12.0+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 5060
GPU Memory: 8.5 GB


## 3. Configuration

In [67]:
# SN32 specific constants
SN32_MAX_LENGTH = 1024  # Must match ctx1024 in model name
MODEL_NAME = "microsoft/deberta-v3-large"

# Optimized config for speed
config = {
    "output_dir": "./fine_tuned_sn32_model",
    "batch_size": 8,
    "gradient_accumulation": 4,  # Reduce to keep effective batch 16
    "learning_rate": 1.5e-5,
    "warmup_ratio": 0.1,
    "epochs": 3,
    "use_lora": True,
    "lora_r": 4,
    "max_length": 256,  # Reduce from 384 (faster tokenization)
}

print("Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

Configuration:
  output_dir: ./fine_tuned_sn32_model
  batch_size: 8
  gradient_accumulation: 4
  learning_rate: 1.5e-05
  warmup_ratio: 0.1
  epochs: 3
  use_lora: True
  lora_r: 4
  max_length: 256


## 4. Load Dataset

In [68]:
# Use only 100,000 samples for faster training
SAMPLE_SIZE = 100000  # Adjust this as needed

print(f"Loading dataset from HuggingFace...")
dataset = load_dataset("../models/ahmadreza13_human-vs-Ai-generated-dataset/data", split="train")

print(f"Full dataset size: {len(dataset):,} examples")

# SHUFFLE FIRST before sampling (critical!)
print("Shuffling dataset...")
dataset = dataset.shuffle(seed=42)

# Then take samples
dataset = dataset.select(range(min(SAMPLE_SIZE, len(dataset))))

# Rename label column
dataset = dataset.rename_column("generated", "label")

print(f"Sampled dataset size: {len(dataset):,} examples")

# Check class distribution
from collections import Counter
labels = [item["label"] for item in dataset]
label_counts = Counter(labels)
print(f"\nClass distribution:")
print(f"  Human (0): {label_counts.get(0, 0):,} ({label_counts.get(0, 0)/len(dataset)*100:.1f}%)")
print(f"  AI (1): {label_counts.get(1, 0):,} ({label_counts.get(1, 0)/len(dataset)*100:.1f}%)")

# Show sample (now should show both classes)
print("\nSample data (after shuffling):")
for i in range(5):
    label_text = 'AI' if dataset[i]['label'] else 'Human'
    print(f"  {dataset[i]['data'][:80]}... -> {label_text}")

Loading dataset from HuggingFace...
Full dataset size: 3,614,247 examples
Shuffling dataset...
Sampled dataset size: 100,000 examples

Class distribution:
  Human (0): 57,634 (57.6%)
  AI (1): 42,366 (42.4%)

Sample data (after shuffling):
  Brzeźno, Toruń County

Brzeźno is a village in the administrative district of Gm... -> Human
  Yes, there is a difference between a tarot card reader and a psychic. A tarot ca... -> AI
  Warren Cup

The Warren Cup is an ancient Greco-Roman silver drinking cup decorat... -> Human
  Hazelle P. Rogers

Hazelle P. Rogers (born September 28, 1952) is a former Democ... -> Human
  Disabled veteran street vendors

Disabled veteran street vendors in New York Cit... -> Human


## 5. Train/Validation Split

In [69]:
print("Splitting into train (95%) and validation (5%)...")

# Use manual stratification
from collections import defaultdict
import random

# Get all indices by label (use 'dataset' directly, not dataset["train"])
label_indices = defaultdict(list)
for idx, item in enumerate(dataset):  # Changed from dataset["train"] to dataset
    label_indices[item["label"]].append(idx)

# Split indices for each label
train_indices = []
val_indices = []

for label, indices in label_indices.items():
    random.seed(42)
    random.shuffle(indices)
    split_point = int(len(indices) * 0.95)
    train_indices.extend(indices[:split_point])
    val_indices.extend(indices[split_point:])

# Shuffle the indices
random.shuffle(train_indices)
random.shuffle(val_indices)

# Create new splits
dataset_split = DatasetDict({
    "train": dataset.select(train_indices),  # Changed from dataset["train"] to dataset
    "validation": dataset.select(val_indices)  # Changed from dataset["train"] to dataset
})

print(f"Train size: {len(dataset_split['train']):,}")
print(f"Validation size: {len(dataset_split['validation']):,}")

Splitting into train (95%) and validation (5%)...
Train size: 94,999
Validation size: 5,001


## 6. Tokenizer

In [70]:
# Load tokenizer FIRST
from transformers import AutoTokenizer

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("../models/deberta-v3-large-hf-weights")
print("✓ Tokenizer loaded")

# Now tokenize
print("Tokenizing dataset with labels...")

def tokenize_function(examples):
    # Tokenize the texts
    tokenized = tokenizer(
        examples["data"],
        truncation=True,
        padding=False,
        max_length=config['max_length'],
        return_tensors=None
    )
    # Add the labels
    tokenized["labels"] = examples["label"]
    return tokenized

# Tokenize the dataset
tokenized_dataset = dataset_split.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset_split["train"].column_names,
    desc="Tokenizing",
    keep_in_memory=False,
)

print(f"✓ Tokenization complete")
print(f"✓ Dataset has columns: {tokenized_dataset['train'].column_names}")
print(f"✓ 'labels' in dataset: {'labels' in tokenized_dataset['train'].column_names}")

Loading tokenizer...


/home/ine/miniconda3/envs/bittensor/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:473: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


✓ Tokenizer loaded
Tokenizing dataset with labels...
✓ Tokenization complete
✓ Dataset has columns: ['input_ids', 'token_type_ids', 'attention_mask', 'labels']
✓ 'labels' in dataset: True


## 7. Load Model with Optional LoRA

In [89]:
## 7. Load Model with Optional LoRA (FIXED)

from transformers import AutoModel, AutoConfig
import torch.nn as nn

LOCAL_MODEL_PATH = "../models/deberta-v3-large-hf-weights"

print(f"Loading base model from LOCAL path: {LOCAL_MODEL_PATH}")

# Load the base model (without classification head)
base_model = AutoModel.from_pretrained(
    LOCAL_MODEL_PATH,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

# Add a classification head WITH gradient checkpointing support
class DebertaForClassification(nn.Module):
    def __init__(self, base_model, num_labels=2):
        super().__init__()
        self.base_model = base_model
        self.classifier = nn.Linear(base_model.config.hidden_size, num_labels)
        
    def forward(self, input_ids, attention_mask=None, labels=None, **kwargs):
        outputs = self.base_model(input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0, :]  # Use [CLS] token
        logits = self.classifier(pooled)
        
        # Return a tuple-like object that Trainer expects
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)
            return (loss, logits)
        return (logits,)
    
   # Replace the gradient_checkpointing_enable method in your model class with:

    def gradient_checkpointing_enable(self, gradient_checkpointing_kwargs=None):
        """Enable gradient checkpointing for the base model"""
        if hasattr(self.base_model, 'gradient_checkpointing_enable'):
            # Pass the kwargs if the method accepts them
            if gradient_checkpointing_kwargs:
                self.base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs=gradient_checkpointing_kwargs)
            else:
                self.base_model.gradient_checkpointing_enable()
        else:
            # For DeBERTa, try to set the attribute directly
            self.base_model.config.gradient_checkpointing = True
            if hasattr(self.base_model, 'encoder'):
                self.base_model.encoder.gradient_checkpointing = True
    
    def gradient_checkpointing_disable(self):
        """Disable gradient checkpointing"""
        if hasattr(self.base_model, 'gradient_checkpointing_disable'):
            self.base_model.gradient_checkpointing_disable()
        else:
            self.base_model.config.gradient_checkpointing = False
            if hasattr(self.base_model, 'encoder'):
                self.base_model.encoder.gradient_checkpointing = False

model = DebertaForClassification(base_model, num_labels=2)

print("✓ Model loaded from local files!")

# Apply LoRA
if config['use_lora']:
    from peft import LoraConfig, get_peft_model, TaskType
    
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=config['lora_r'],
        lora_alpha=32,
        target_modules=["query_proj", "key_proj", "value_proj", "dense"],
        lora_dropout=0.1,
        bias="none",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print(f"Model on device: {device}")

Loading base model from LOCAL path: ../models/deberta-v3-large-hf-weights
✓ Model loaded from local files!
trainable params: 1,771,522 || all params: 435,785,732 || trainable%: 0.40651216180707817
Model on device: cuda


## 8. Metrics Function (SN32 Compatible)

In [81]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, average_precision_score

def compute_metrics(eval_pred):
    """Compute SN32 reward metrics (matches validator.reward() exactly)"""
    predictions, labels = eval_pred
    probs = torch.nn.functional.softmax(torch.from_numpy(predictions), dim=-1)
    preds = np.argmax(predictions, axis=1)
    
    # Calculate metrics
    accuracy = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='binary')
    ap_score = average_precision_score(labels, probs[:, 1])
    
    # False Positive Rate (FP Score) - critical for SN32
    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    fp_score = 1 - (fp / len(labels)) if len(labels) > 0 else 0
    
    # SN32 reward calculation (same as validator)
    sn32_reward = (f1 + fp_score + ap_score) / 3
    
    return {
        "accuracy": accuracy,
        "f1_score": f1,
        "ap_score": ap_score,
        "fp_score": fp_score,
        "sn32_reward": sn32_reward,
    }

# Test with sample data (optional)
print("Testing metrics function...")
test_preds = np.array([[0.1, 0.9], [0.8, 0.2], [0.3, 0.7], [0.9, 0.1]])
test_labels = np.array([1, 0, 1, 0])
results = compute_metrics((test_preds, test_labels))
print(f"Sample results: {results}")

Testing metrics function...
Sample results: {'accuracy': 1.0, 'f1_score': np.float64(1.0), 'ap_score': np.float64(1.0), 'fp_score': np.float64(1.0), 'sn32_reward': np.float64(1.0)}


## 9. Training Arguments

In [82]:
# Calculate effective batch size
effective_batch = config['batch_size'] * config['gradient_accumulation']
print(f"Effective batch size: {effective_batch}")

# Training arguments - WITHOUT gradient_checkpointing
training_args = TrainingArguments(
    output_dir=config['output_dir'],
    per_device_train_batch_size=config['batch_size'],
    per_device_eval_batch_size=config['batch_size'] * 2,
    gradient_accumulation_steps=config['gradient_accumulation'],
    learning_rate=config['learning_rate'],
    warmup_ratio=config['warmup_ratio'],
    weight_decay=0.01,
    max_grad_norm=1.0,
    num_train_epochs=config['epochs'],
    logging_dir=f"{config['output_dir']}/logs",
    logging_steps=50,
    logging_strategy="steps",
    evaluation_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="sn32_reward",
    greater_is_better=True,
    push_to_hub=False,
    fp16=torch.cuda.is_available(),
    report_to="none",
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    remove_unused_columns=False,
    # gradient_checkpointing=True,  # <-- MAKE SURE THIS IS REMOVED OR COMMENTED
    seed=42,
)

print("\n" + "="*50)
print("TRAINING CONFIGURATION")
print("="*50)
print(f"  Total training samples: {len(tokenized_dataset['train']):,}")
print(f"  Batch size (effective): {effective_batch}")
total_steps = (len(tokenized_dataset['train']) // effective_batch) * config['epochs']
print(f"  Total steps: ~{total_steps:,}")
print(f"  Expected time: ~{total_steps * 0.3 / 60:.1f} hours")
print(f"  FP16: {training_args.fp16}")
print(f"  Gradient checkpointing: DISABLED")
print(f"  Metric: {training_args.metric_for_best_model}")
print("="*50)

print("\n✓ Training arguments created successfully!")

Effective batch size: 32

TRAINING CONFIGURATION
  Total training samples: 94,999
  Batch size (effective): 32
  Total steps: ~8,904
  Expected time: ~44.5 hours
  FP16: True
  Gradient checkpointing: DISABLED
  Metric: sn32_reward

✓ Training arguments created successfully!


## 10. Initialize Trainer

In [83]:
data_collator = DataCollatorWithPadding(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("Trainer initialized successfully!")

Trainer initialized successfully!


/home/ine/miniconda3/envs/bittensor/lib/python3.10/site-packages/accelerate/accelerator.py:439: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  if kwargs_handlers is not None:


## 11. Train the Model

In [84]:
# Verify trainer is properly initialized
print("Verifying trainer setup...")

# Check if trainer exists
try:
    print(f"✓ Trainer object exists: {trainer is not None}")
except NameError:
    print("❌ Trainer not defined! Please create trainer first.")
    print("   Run the Trainer initialization cell before training.")
    
# Check if model is loaded
try:
    print(f"✓ Model loaded: {model is not None}")
    print(f"✓ Model device: {next(model.parameters()).device}")
except:
    print("❌ Model not loaded!")
    
# Check if datasets exist
try:
    print(f"✓ Train dataset: {len(tokenized_dataset['train']):,} samples")
    print(f"✓ Validation dataset: {len(tokenized_dataset['validation']):,} samples")
except:
    print("❌ Datasets not ready!")

# Check GPU
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"✓ GPU Memory available: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1e9:.1f} GB")
else:
    print("⚠️ WARNING: CUDA not available - training will be very slow!")

print("\n✓ Verification complete!")

Verifying trainer setup...
✓ Trainer object exists: True
✓ Model loaded: True
✓ Model device: cuda:0
✓ Train dataset: 94,999 samples
✓ Validation dataset: 5,001 samples
✓ GPU: NVIDIA GeForce RTX 5060
✓ GPU Memory: 8.5 GB
✓ GPU Memory available: -5.7 GB

✓ Verification complete!


In [85]:
from transformers import TrainerCallback

class LoggingCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            step = logs.get("step", state.global_step)
            loss = logs.get("loss", None)
            eval_loss = logs.get("eval_loss", None)
            f1 = logs.get("eval_f1_score", None)
            sn32 = logs.get("eval_sn32_reward", None)
            
            if loss:
                print(f"\n[Step {step}] Training Loss: {loss:.4f}")
            if eval_loss:
                print(f"[Step {step}] Eval Loss: {eval_loss:.4f}")
            if f1:
                print(f"[Step {step}] F1 Score: {f1:.4f}")
            if sn32:
                print(f"[Step {step}] SN32 Reward: {sn32:.4f}")
                print("-" * 50)

# Add callback to trainer
trainer.add_callback(LoggingCallback())
print("✓ Logging callback added")

✓ Logging callback added


In [91]:
## 11b. Ultra-Conservative Training

import gc
import torch
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

gc.collect()
torch.cuda.empty_cache()

# Use very small subset
SMALL_SUBSET = 500
train_small = tokenized_dataset['train'].select(range(SMALL_SUBSET))

# Override trainer with conservative settings
trainer.train_dataset = train_small

# Force small batch size
training_args.per_device_train_batch_size = 2
training_args.per_device_eval_batch_size = 2
training_args.gradient_accumulation_steps = 4  # Effective batch = 8

print(f"Training on {SMALL_SUBSET} samples")
print(f"Batch size: 2, Gradient accumulation: 4 (effective: 8)")
print("\nStarting training...")

try:
    result = trainer.train()
    print("✓ Training successful!")
except Exception as e:
    print(f"Error: {e}")
    print("\nIf this fails, restart kernel and run cells 1-9 only, then this cell")

Training on 500 samples
Batch size: 2, Gradient accumulation: 4 (effective: 8)

Starting training...


You're using a DebertaV2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a DebertaV2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Error: 'DebertaForClassification' object has no attribute 'config'

If this fails, restart kernel and run cells 1-9 only, then this cell


In [90]:
## 11. Train the Model (FIXED)

# Optimized training - Copy and run this entire cell
import gc
import torch

# Clear memory
gc.collect()
torch.cuda.empty_cache()

print("🚀 Starting optimized training...")
print(f"Free GPU memory: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1e9:.2f} GB")

# SAFELY enable gradient checkpointing for memory efficiency
try:
    model.gradient_checkpointing_enable()
    print("✓ Gradient checkpointing enabled")
except AttributeError:
    print("⚠️ Gradient checkpointing not available, continuing without it")
    # Alternative: reduce batch size or max_length if memory issues occur

# Use subset for faster training (adjust as needed)
SUBSET_SIZE = 5000  # Train on 5k samples first - increase if you want
if len(tokenized_dataset['train']) > SUBSET_SIZE:
    print(f"Using {SUBSET_SIZE:,} samples (from {len(tokenized_dataset['train']):,})")
    train_dataset = tokenized_dataset['train'].select(range(SUBSET_SIZE))
else:
    train_dataset = tokenized_dataset['train']

# Update training args (only if gradient_checkpointing attribute exists)
if hasattr(training_args, 'gradient_checkpointing'):
    training_args.gradient_checkpointing = True
training_args.dataloader_num_workers = 2  # Reduced from 4 to avoid issues
training_args.dataloader_prefetch_factor = 2 if hasattr(training_args, 'dataloader_prefetch_factor') else None

# Update trainer with subset
trainer.train_dataset = train_dataset

print("\n" + "="*60)
print("Starting training...")
print("="*60)
print(f"Training samples: {len(train_dataset):,}")
print(f"Batch size: {config['batch_size']}")
print(f"Steps per epoch: {len(train_dataset) // (config['batch_size'] * config['gradient_accumulation']):,}")
print(f"Total steps: {len(train_dataset) // (config['batch_size'] * config['gradient_accumulation']) * config['epochs']:,}")
print("="*60)

# Train with error handling
try:
    train_result = trainer.train()
    print("\n" + "="*60)
    print("✓ Training completed!")
    print("="*60)
    
    # Save model
    trainer.save_model(config['output_dir'])
    tokenizer.save_pretrained(config['output_dir'])
    print(f"✓ Model saved to {config['output_dir']}")
except Exception as e:
    print(f"\n❌ Training failed with error: {e}")
    print("\nTroubleshooting suggestions:")
    print("1. Reduce SUBSET_SIZE (try 1000 instead of 5000)")
    print("2. Reduce batch_size to 4 or 2")
    print("3. Reduce max_length to 128")
    print("4. Disable gradient_checkpointing by commenting out the try block above")

🚀 Starting optimized training...
Free GPU memory: -5.65 GB
✓ Gradient checkpointing enabled
Using 5,000 samples (from 94,999)

Starting training...
Training samples: 5,000
Batch size: 8
Steps per epoch: 156
Total steps: 468


You're using a DebertaV2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a DebertaV2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.



❌ Training failed with error: 'DebertaForClassification' object has no attribute 'config'

Troubleshooting suggestions:
1. Reduce SUBSET_SIZE (try 1000 instead of 5000)
2. Reduce batch_size to 4 or 2
3. Reduce max_length to 128
4. Disable gradient_checkpointing by commenting out the try block above


In [ ]:
# Monitor training progress (run this while training)
import time
import psutil

print("📊 Training Monitor")
print("="*50)

# Get GPU stats
if torch.cuda.is_available():
    print(f"GPU Memory: {torch.cuda.memory_allocated(0)/1e9:.2f} GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    
# Get CPU/RAM stats
print(f"RAM: {psutil.virtual_memory().used/1e9:.1f} GB / {psutil.virtual_memory().total/1e9:.1f} GB")
print(f"CPU: {psutil.cpu_percent()}%")

## 12. Save Model in SN32 Format

In [ ]:
print(f"Saving model to {config['output_dir']}")
trainer.save_model(config['output_dir'])
tokenizer.save_pretrained(config['output_dir'])

# Convert to SN32 format (.pth for task weights)
if config['use_lora']:
    from peft import PeftModel
    print("Merging LoRA weights with base model...")
    merged_model = model.merge_and_unload()
    torch.save(merged_model.state_dict(), f"{config['output_dir']}/sn32_deberta_weights.pth")
else:
    # Save just the task-specific weights (smaller file)
    torch.save(model.state_dict(), f"{config['output_dir']}/sn32_deberta_weights.pth")

print(f"✅ SN32 weights saved to: {config['output_dir']}/sn32_deberta_weights.pth")

# Check file size
import os
size_mb = os.path.getsize(f"{config['output_dir']}/sn32_deberta_weights.pth") / (1024 * 1024)
print(f"File size: {size_mb:.1f} MB")

## 13. Evaluate Final Model

In [ ]:
print("Final evaluation on validation set...")
eval_results = trainer.evaluate()

print("\n" + "=" * 50)
print("FINAL RESULTS")
print("=" * 50)
for key, value in eval_results.items():
    if key.startswith("eval_"):
        metric_name = key[5:]
        print(f"{metric_name:20s}: {value:.4f}")

# Check if model meets SN32 requirements
f1 = eval_results.get("eval_f1_score", 0)
fp_score = eval_results.get("eval_fp_score", 0)

print("\n" + "-" * 50)
if f1 >= 0.9 and fp_score >= 0.98:
    print("✅ Model meets SN32 minimum requirements!")
    print(f"   F1 Score: {f1:.4f} (target: 0.90)")
    print(f"   FP Score: {fp_score:.4f} (target: 0.98)")
else:
    print("⚠️ Model does NOT meet SN32 minimum requirements yet")
    print(f"   F1 Score: {f1:.4f} (target: 0.90)")
    print(f"   FP Score: {fp_score:.4f} (target: 0.98)")
    print("   Consider: more epochs, larger model, or better data preprocessing")

# Save metrics
with open(f"{config['output_dir']}/metrics.json", "w") as f:
    json.dump(eval_results, f, indent=2)
print(f"\nMetrics saved to: {config['output_dir']}/metrics.json")

## 14. Visualize Results

In [ ]:
# Load training history if available
if os.path.exists(f"{config['output_dir']}/checkpoint-*/trainer_state.json"):
    import glob
    state_files = glob.glob(f"{config['output_dir']}/checkpoint-*/trainer_state.json")
    if state_files:
        with open(sorted(state_files)[-1], 'r') as f:
            history = json.load(f)
        
        # Extract metrics
        steps = []
        losses = []
        eval_losses = []
        
        for log in history['log_history']:
            if 'loss' in log:
                steps.append(log['step'])
                losses.append(log['loss'])
            if 'eval_loss' in log:
                eval_losses.append((log['step'], log['eval_loss']))
        
        if losses:
            plt.figure(figsize=(12, 4))
            
            plt.subplot(1, 2, 1)
            plt.plot(steps, losses, label='Training Loss')
            if eval_losses:
                eval_steps, eval_vals = zip(*eval_losses)
                plt.plot(eval_steps, eval_vals, label='Validation Loss')
            plt.xlabel('Steps')
            plt.ylabel('Loss')
            plt.title('Training Progress')
            plt.legend()
            plt.grid(True)
            
            plt.subplot(1, 2, 2)
            metrics_to_plot = ['f1_score', 'fp_score', 'sn32_reward']
            for metric in metrics_to_plot:
                values = []
                metric_steps = []
                for log in history['log_history']:
                    if f'eval_{metric}' in log:
                        metric_steps.append(log['step'])
                        values.append(log[f'eval_{metric}'])
                if values:
                    plt.plot(metric_steps, values, label=metric)
            plt.xlabel('Steps')
            plt.ylabel('Score')
            plt.title('SN32 Metrics')
            plt.legend()
            plt.grid(True)
            
            plt.tight_layout()
            plt.savefig(f"{config['output_dir']}/training_curves.png", dpi=150)
            plt.show()
            print(f"Training curves saved to: {config['output_dir']}/training_curves.png")
else:
    print("Training history not available for visualization")

## 15. Deploy to SN32 Miner

In [ ]:
print("=" * 50)
print("DEPLOYMENT INSTRUCTIONS")
print("=" * 50)
print("""
To deploy your fine-tuned model to SN32 miner:

1. Copy the weights to your models directory:
   ```bash
   # Backup original
   mv models/deberta-large-ls03-ctx1024.pth models/deberta-large-ls03-ctx1024.pth.backup
   
   # Copy your fine-tuned weights
   cp ./fine_tuned_sn32_model/sn32_deberta_weights.pth models/deberta-large-ls03-ctx1024.pth
   ```

2. Restart your miner:
   ```bash
   pm2 restart net32-miner
   # or
   pm2 restart llm_detection_miners
   ```

3. Monitor performance:
   ```bash
   pm2 logs net32-miner
   ```

4. Check your scores on:
   - https://wandb.ai/itsai-dev/subnet32
   - https://huggingface.co/spaces/sergak0/ai-detection-leaderboard
""")

# Optional: Test inference with a few samples
print("\nTesting inference with trained model...")
model.eval()
test_texts = [
    "The quick brown fox jumps over the lazy dog.",  # Human-like
    "I am an AI language model designed to assist users with various tasks.",  # AI-like
]

for text in test_texts:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        ai_prob = probs[0][1].item()
    print(f"Text: {text[:50]}...")
    print(f"  AI Probability: {ai_prob:.3f}")
    print(f"  Prediction: {'AI' if ai_prob > 0.5 else 'Human'}\n")

## 16. Optional: Upload to HuggingFace Hub

In [ ]:
# Uncomment to upload your model to HuggingFace
# from huggingface_hub import notebook_login
# notebook_login()

# model.push_to_hub("your-username/sn32-deberta-ai-detector")
# tokenizer.push_to_hub("your-username/sn32-deberta-ai-detector")
# print("Model uploaded to HuggingFace Hub!")